## AI Text Detection ##

We run Pangram on all cleaned sampled texted across all sampled candidates. The final dataframe returns the given CSV with the additional columns of label_final, ai_assistance_final, confidence_final, fraction_ai, fraction_ai_assisted, fraction_human, and num_ai_segments; all of these scores are obtained with the Pangram API using research credits

In [1]:
import pandas as pd
from pangram import Pangram
import numpy as np
import random
import re
from collections import Counter #for extracting labels and confidence majorities with pangram
import os
from dotenv import load_dotenv

In [2]:
governor_data = pd.read_csv("../governor_content.csv")

In [3]:
ag_data = pd.read_csv("../ag_content.csv")

In [7]:
mayor_data = pd.read_csv("../mayor_content.csv")

In [9]:
us_senate_data = pd.read_csv("../us_senate_content.csv")

In [172]:
us_house_data = pd.read_csv("../us_house_content.csv")

In [174]:
state_house_data = pd.read_csv("../state_house_content.csv")

In [175]:
state_senate_data = pd.read_csv("../state_senate_content.csv")

In [57]:
load_dotenv()
pangram_client = Pangram(api_key=os.getenv("PANGRAM_API_KEY"))

### Data Cleaning ###
This notebook uses data queried from camplinks.db from the Content table to get the sampled text for all candidates in our database.This is loaded in as "data". We then clean the dataframe by dropping rows where the unprocessed_text and sampled_text are both NA or where sampled_text = ERROR: could not fetch page. 

The dataframe also contains features of candidates that have been combined from the camplinks.db, allowing for easier future analysis using only one CSV

In [11]:
#add columns to datasets to fill in with pangram results

def add_columns(df):
    columns_to_add = [
        'AI_label',
        'assistance_score',
        'confidence',
        'fraction_ai',
        'fraction_ai_assisted',
        'fraction_human',
        'num_ai_segments'
    ]
    
    for col in columns_to_add:
        df[col] = float('nan')

    return df


def clean_data(df):
    df_filtered = df[~(df['unprocessed_text'].isna() & df['sampled_text'].isna())]
    df_cleaned = df_filtered[df_filtered['sampled_text'] != 'ERROR: could not fetch page']
    return df_cleaned

In [13]:
#some exploratory functions:

def avg_word_count(df):
    return df['sampled_text'].str.split().str.len().mean()

def cand_amount(df):
    return df['candidate_name'].nunique()

Clean and add columns:

In [15]:
gov_cleaned = clean_data(governor_data)
gov_added = add_columns(gov_cleaned)

In [17]:
ag_cleaned = clean_data(ag_data)
ag_added = add_columns(ag_cleaned)

In [19]:
mayor_cleaned = clean_data(mayor_data)
mayor_added = add_columns(mayor_cleaned)

In [21]:
us_senate_cleaned = clean_data(us_senate_data)
us_senate_added = add_columns(us_senate_cleaned)

In [192]:
us_house_cleaned = clean_data(us_house_data)
us_house_added = add_columns(us_house_cleaned)

In [194]:
state_senate_cleaned = clean_data(state_senate_data)
state_senate_added = add_columns(state_senate_cleaned)

In [196]:
state_house_cleaned = clean_data(state_house_data)
state_house_added = add_columns(state_house_cleaned)

Get some exploratory information

In [198]:
#exploratory info
print('governor candidates:', cand_amount(gov_added), '- average word count and text samples: ', avg_word_count(gov_added) , ', ' , len(gov_added))
print('attorney general candidates:', cand_amount(ag_added), '- average word count and text samples: ', avg_word_count(ag_added), ', ', len(ag_added))
print('mayoral candidates:', cand_amount(mayor_added), '- average word count and text samples: ', avg_word_count(mayor_added), ', ', len(mayor_added))
print('US senate candidates:', cand_amount(us_senate_added), '- average word count and text samples: ', avg_word_count(us_senate_added), ', ', len(us_senate_added))
print('US house candidates:', cand_amount(us_house_added), '- average word count and text samples: ', avg_word_count(us_house_added), ', ', len(us_house_added))
print('State senate candidates:', cand_amount(state_senate_added), '- average word count and text samples:  ', avg_word_count(state_senate_added), ', ', len(state_senate_added))
print('State house candidates:', cand_amount(state_house_added), ' - average word count and text samples: ', avg_word_count(state_house_added), ', ', len(state_house_added))

governor candidates: 29 - average word count and text samples:  242.83582089552237 ,  67
attorney general candidates: 26 - average word count and text samples:  206.02325581395348 ,  43
mayoral candidates: 155 - average word count and text samples:  281.3973941368078 ,  307
US senate candidates: 58 - average word count and text samples:  237.9624060150376 ,  133
US house candidates: 809 - average word count and text samples:  292.2930513595166 ,  1986
State senate candidates: 1469 - average word count and text samples:   242.05026769779892 ,  3362
State house candidates: 6615  - average word count and text samples:  238.12985064892715 ,  15333


In [200]:
#combine all tables together to view across all races (this is just exploratory, detection will be ran on individual dataframes(
combined = pd.concat([gov_added, ag_added, mayor_added, us_senate_added, us_house_added, state_senate_added, state_house_added], ignore_index=True)
combined.head()

,content_id,candidate_id,candidate_name,page_url,page_type,link_type,unprocessed_text,cleaned_text,sampled_text,race_type,year,AI_label,assistance_score,confidence,fraction_ai,fraction_ai_assisted,fraction_human,num_ai_segments
0,1,6179,Andy Beshear,https://andybeshear.com/,home,campaign_site,Andy Beshear Menu Meet Andy Meet Jacqueline An...,Andy Beshear Menu Meet Andy Meet Jacqueline An...,Andy Beshear Menu Meet Andy Meet Jacqueline An...,Governor,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,6180,Daniel Cameron,https://cameronforkentucky.com/,home,campaign_site,Menu News Facebook Twitter Instagram Donate › ...,Menu News Facebook Twitter Instagram Donate S...,I'm In! QUICK DONATE $25 $50 $100 ENTER AMOUNT...,Governor,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,6180,Daniel Cameron,https://cameronforkentucky.com/policies/terms,policy,campaign_site,Menu News Facebook Twitter Instagram Donate › ...,Menu News Facebook Twitter Instagram Donate S...,Data Privacy We will not share or sell your pe...,Governor,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,6180,Daniel Cameron,https://cameronforkentucky.com/policies/privacy,policy,campaign_site,Menu News Facebook Twitter Instagram Donate › ...,Menu News Facebook Twitter Instagram Donate S...,"This Privacy Policy explains how we collect, u...",Governor,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,6180,Daniel Cameron,https://cameronforkentucky.com/issues/,policy,campaign_site,Menu News Facebook Twitter Instagram Donate › ...,Menu News Facebook Twitter Instagram Donate S...,He will always oppose the radical green agenda...,Governor,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [147]:
print('Total candidates:', cand_amount(combined), ' - average word count and text samples: ', avg_word_count(combined), ', ', len(combined))


Total candidates: 9126  - average word count and text samples:  244.39164429372144 ,  21231


In [83]:
combined['page_type'].value_counts() / len(combined)

page_type
home      0.442560
about     0.293533
policy    0.263907
Name: count, dtype: float64

### Running AI Text Detection ###

In [ ]:
#add in empty columns so that we don't accidently lose any pangram data
#instead, automatically append an entire row once its done; additionally run in batches 

In [23]:
#code source from pangram: https://pangram.readthedocs.io/en/latest/api/pangram.html

def check_AI(text):
    valid_API_call = False
    try:
        result = pangram_client.predict(text)
        valid_API_call = True
    except:
        fraction_ai = float('nan')
        fraction_ai_assisted = float('nan')
        fraction_human = float('nan')
        num_ai_segments = float('nan')
        AI_label = float('nan')
        confidence_final = float('nan')
        assistance_score = float('nan')

    if valid_API_call:
        AI_label = result['prediction_short']
        confidence_all = []
        ai_assistance_all = []
        fraction_ai = result['fraction_ai']
        fraction_ai_assisted = result['fraction_ai_assisted']
        fraction_human = result['fraction_human']
        num_ai_segments = result['num_ai_segments']

        #for some reason, confidence is only per-window. So to get it we need to average out window confidence. Though, there tends
        #to be only 1 or 2 windows so not big deal
        for window in result['windows']:
            confidence = window['confidence']
            confidence_all.append(confidence)

            ai_assistance_score = window['ai_assistance_score'] #same with these
            ai_assistance_all.append(ai_assistance_score)
            
        confidence = Counter(confidence_all).most_common(1)[0][0]
        assistance_score =  np.mean(ai_assistance_all)

    else:
        return ['unable to run', 'unable to run', 'unable to run', 'unable to run', 'unable to run', 'unable to run', 'unable to run']

    #print(AI_label, confidence, fraction_ai, fraction_ai_assisted, fraction_human, num_ai_segments)
    return [AI_label, assistance_score, confidence, fraction_ai, fraction_ai_assisted, fraction_human, num_ai_segments]

In [94]:
def check_entire_race(race_df):
    for idx, row in race_df.iterrows():
        try:
            result = check_AI(row['cleaned_text'])
            print(row['candidate_name'], result)
            race_df.loc[idx, ['AI_label', 'assistance_score', 'confidence', 'fraction_ai', 'fraction_ai_assisted', 'fraction_human', 'num_ai_segments']] = result
        except Exception as e:
            print(f"Row {idx} failed: {e}")
            # row stays None/NaN, loop continues

In [153]:
#check_entire_race(us_senate_added)

Kari Lake ['Human', 0.058674976229667664, 'Medium', 0.0, 0.0, 1.0, 0]


/var/folders/17/gz6b52jj5cvfsq17xw_vghsh0000gn/T/ipykernel_64413/2149402114.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Human' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  race_df.loc[idx, ['AI_label', 'assistance_score', 'confidence', 'fraction_ai', 'fraction_ai_assisted', 'fraction_human', 'num_ai_segments']] = result
/var/folders/17/gz6b52jj5cvfsq17xw_vghsh0000gn/T/ipykernel_64413/2149402114.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Medium' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  race_df.loc[idx, ['AI_label', 'assistance_score', 'confidence', 'fraction_ai', 'fraction_ai_assisted', 'fraction_human', 'num_ai_segments']] = result


Kari Lake ['unable to run', 'unable to run', 'unable to run', 'unable to run', 'unable to run', 'unable to run', 'unable to run']


/var/folders/17/gz6b52jj5cvfsq17xw_vghsh0000gn/T/ipykernel_64413/2149402114.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'unable to run' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  race_df.loc[idx, ['AI_label', 'assistance_score', 'confidence', 'fraction_ai', 'fraction_ai_assisted', 'fraction_human', 'num_ai_segments']] = result
/var/folders/17/gz6b52jj5cvfsq17xw_vghsh0000gn/T/ipykernel_64413/2149402114.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'unable to run' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  race_df.loc[idx, ['AI_label', 'assistance_score', 'confidence', 'fraction_ai', 'fraction_ai_assisted', 'fraction_human', 'num_ai_segments']] = result
/var/folders/17/gz6b52jj5cvfsq17xw_vghsh0000gn/T/ipykernel_64413/2

Kari Lake ['Human', 0.014941799454391003, 'High', 0.0, 0.0, 1.0, 0]
Ruben Gallego ['Human', 0.005045536223665944, 'High', 0.0, 0.0, 1.0, 0]
Ruben Gallego ['Human', 0.010151629336178303, 'High', 0.0, 0.0, 1.0, 0]
Ruben Gallego ['Human', 0.006716141011565924, 'High', 0.0, 0.0, 1.0, 0]
Ruben Gallego ['unable to run', 'unable to run', 'unable to run', 'unable to run', 'unable to run', 'unable to run', 'unable to run']
Ruben Gallego ['Human', 0.004454157315194607, 'High', 0.0, 0.0, 1.0, 0]
Ruben Gallego ['Human', 0.008216402493417263, 'High', 0.0, 0.0, 1.0, 0]
Ruben Gallego ['Human', 0.005655944813042879, 'High', 0.0, 0.0, 1.0, 0]
Ruben Gallego ['Human', 0.00491423299536109, 'High', 0.0, 0.0, 1.0, 0]
Adam B. Schiff ['Human', 0.007768214214593172, 'High', 0.0, 0.0, 1.0, 0]
Adam B. Schiff ['unable to run', 'unable to run', 'unable to run', 'unable to run', 'unable to run', 'unable to run', 'unable to run']
Adam B. Schiff ['Human', 0.0053487834923614105, 'High', 0.0, 0.0, 1.0, 0]
Steve Garvey 

In [154]:
#export done races:

#governor: DONE
#gov_added.to_csv("governor_text_detection.csv", index=False)

#attorney general: DONE
#ag_added.to_csv("ag_text_detection.csv", index=False)

#mayor: DONE
#mayor_added.to_csv("mayor_text_detection.csv", index=False)

#US Senate: DONE
#us_senate_added.to_csv("us_senate_detection.csv",index=False)

#US House

#State Senate

#State House

# Testing different sample lengths #

We originally sampled 40% of the visible text to run Pangram on it. Now, I will sample 60% and 80% of the original text and see if it makes a significant difference in the amount of text identified as AI. 

In [31]:
combined_60 = pd.concat([gov_added, ag_added, mayor_added, us_senate_added], ignore_index=True)
combined_60

,content_id,candidate_id,candidate_name,page_url,page_type,link_type,unprocessed_text,cleaned_text,sampled_text,race_type,year,AI_label,assistance_score,confidence,fraction_ai,fraction_ai_assisted,fraction_human,num_ai_segments
0,1,6179,Andy Beshear,https://andybeshear.com/,home,campaign_site,Andy Beshear Menu Meet Andy Meet Jacqueline An...,Andy Beshear Menu Meet Andy Meet Jacqueline An...,Andy Beshear Menu Meet Andy Meet Jacqueline An...,Governor,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,6180,Daniel Cameron,https://cameronforkentucky.com/,home,campaign_site,Menu News Facebook Twitter Instagram Donate › ...,Menu News Facebook Twitter Instagram Donate S...,I'm In! QUICK DONATE $25 $50 $100 ENTER AMOUNT...,Governor,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,6180,Daniel Cameron,https://cameronforkentucky.com/policies/terms,policy,campaign_site,Menu News Facebook Twitter Instagram Donate › ...,Menu News Facebook Twitter Instagram Donate S...,Data Privacy We will not share or sell your pe...,Governor,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,6180,Daniel Cameron,https://cameronforkentucky.com/policies/privacy,policy,campaign_site,Menu News Facebook Twitter Instagram Donate › ...,Menu News Facebook Twitter Instagram Donate S...,"This Privacy Policy explains how we collect, u...",Governor,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,6180,Daniel Cameron,https://cameronforkentucky.com/issues/,policy,campaign_site,Menu News Facebook Twitter Instagram Donate › ...,Menu News Facebook Twitter Instagram Donate S...,He will always oppose the radical green agenda...,Governor,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
545,2828,6155,Tammy Baldwin,https://www.tammybaldwin.com/about/,about,campaign_site,About Tammy News Issues Get Involved Store Con...,About Tammy News Issues Get Involved Store Con...,"James Allen Veteran Vision Equity Act, passed ...",US Senate,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN
546,2829,6158,Glenn Elliott,https://www.elliottforwv.com,home,campaign_site,Elliott Wave Forecast Login Start 14-Day Trial...,Elliott Wave Forecast Login Start 14-Day Trial...,Retail Traders & Investors Join us never trade...,US Senate,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN
547,2830,6157,Jim Justice,http://jimjusticewv.com,home,campaign_site,Skip to content Endorsed by Trump Meet Jim Acc...,Skip to content Endorsed by Trump Meet Jim Acc...,Message frequency may vary. Reply stop to stop...,US Senate,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN
548,2831,6157,Jim Justice,https://jimjusticewv.com/issues/,policy,campaign_site,Skip to content Endorsed by Trump Meet Jim Acc...,Skip to content Endorsed by Trump Meet Jim Acc...,Reply stop to stop and help for help. Mobile o...,US Senate,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [70]:
#use the same sampling method but fraction is 0.6
def _sample_text(text: str, fraction: float = 0.8, max_attempts: int = 5) -> str:
    """Return a contiguous ~40% sentence chunk starting at a random position.

    Retries up to max_attempts times to avoid chunks containing ``"ERROR"``.

    Args:
        text: Input text to sample from.
        fraction: Fraction of sentences to include.
        max_attempts: Maximum retry attempts to avoid ERROR-containing chunks.

    Returns:
        Sampled text string.
    """
    if not isinstance(text, str) or text.startswith("ERROR:") or text == "":
        return text
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
    if not sentences:
        return text
    k = max(1, int(len(sentences) * fraction))
    max_start = max(0, len(sentences) - k)
    sample = ""
    for _ in range(max_attempts):
        start = random.randint(0, max_start)
        sample = " ".join(sentences[start : start + k])
        if "ERROR" not in sample:
            return sample
    return sample

In [37]:
combined_60['sampled_text_60'] = np.nan

In [43]:
combined_60['sampled_text_60'] = combined_60['cleaned_text'].apply(_sample_text)

In [80]:
combined_60['sampled_text_60'].str.split().str.len().mean()

394.7490909090909

In [59]:
#check_entire_race(combined_60)

Andy Beshear ['Human', 0.005448558833450079, 'High', 0.0, 0.0, 1.0, 0]
Daniel Cameron ['Human', 0.006664706394076347, 'High', 0.0, 0.0, 1.0, 0]
Daniel Cameron ['AI', 0.8605841576390795, 'Medium', 1.0, 0.0, 0.0, 1]
Daniel Cameron ['AI', 0.989897112051646, 'High', 1.0, 0.0, 0.0, 2]
Daniel Cameron ['Human', 0.2740520238876343, 'Low', 0.0, 0.0, 1.0, 0]
Jeff Landry ['Human', 0.0055374037474393845, 'High', 0.0, 0.0, 1.0, 0]
Jeff Landry ['Human', 0.01942707778936793, 'High', 0.0, 0.0, 1.0, 0]
Brandon Presley ['Human', 0.005743841469908754, 'High', 0.0, 0.0, 1.0, 0]
Tate Reeves ['Human', 0.004855262581259012, 'High', 0.0, 0.0, 1.0, 0]
Tate Reeves ['Human', 0.0048487186431884766, 'High', 0.0, 0.0, 1.0, 0]
Matt Meyer ['Human', 0.012467694775555602, 'High', 0.0, 0.0, 1.0, 0]
Michael Ramone ['Human', 0.006591524463146925, 'High', 0.0, 0.0, 1.0, 0]
Michael Ramone ['Human', 0.007156895939260721, 'High', 0.0, 0.0, 1.0, 0]
Michael Ramone ['Human', 0.009534815326333046, 'High', 0.0, 0.0, 1.0, 0]
Jennif

In [63]:
#combined_60.to_csv("combined_60.csv")

80% sampling:

In [68]:
combined_80 = pd.concat([gov_added, ag_added, mayor_added, us_senate_added], ignore_index=True)

In [76]:
combined_80['sampled_text_80'] = np.nan
combined_80['sampled_text_80'] = combined_80['cleaned_text'].apply(_sample_text)

In [78]:
combined_80['sampled_text_80'].str.split().str.len().mean()

527.2963636363636

In [86]:
check_entire_race(combined_80)

Andy Beshear ['Human', 0.006113550625741482, 'High', 0.0, 0.0, 1.0, 0]


/var/folders/17/gz6b52jj5cvfsq17xw_vghsh0000gn/T/ipykernel_25992/2497770618.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Human' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  race_df.loc[idx, ['AI_label', 'assistance_score', 'confidence', 'fraction_ai', 'fraction_ai_assisted', 'fraction_human', 'num_ai_segments']] = result
/var/folders/17/gz6b52jj5cvfsq17xw_vghsh0000gn/T/ipykernel_25992/2497770618.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'High' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  race_df.loc[idx, ['AI_label', 'assistance_score', 'confidence', 'fraction_ai', 'fraction_ai_assisted', 'fraction_human', 'num_ai_segments']] = result


Daniel Cameron ['Human', 0.007358470465987921, 'High', 0.0, 0.0, 1.0, 0]
Daniel Cameron ['AI', 0.9640528308717828, 'High', 1.0, 0.0, 0.0, 2]
Daniel Cameron ['AI', 0.9897981413773128, 'High', 1.0, 0.0, 0.0, 2]
Daniel Cameron ['Human', 0.3512515399532764, 'Low', 0.0, 0.0, 1.0, 0]
Jeff Landry ['Human', 0.006167260347865522, 'High', 0.0, 0.0, 1.0, 0]
Jeff Landry ['Human', 0.01602179515437815, 'High', 0.0, 0.0, 1.0, 0]
Brandon Presley ['Human', 0.00574190810719436, 'High', 0.0, 0.0, 1.0, 0]
Tate Reeves ['Human', 0.004526725970208645, 'High', 0.0, 0.0, 1.0, 0]
Tate Reeves ['Human', 0.005101080518215895, 'High', 0.0, 0.0, 1.0, 0]
Matt Meyer ['Human', 0.011196104794443171, 'High', 0.0, 0.0, 1.0, 0]
Michael Ramone ['Human', 0.010066330432891846, 'High', 0.0, 0.0, 1.0, 0]
Michael Ramone ['Human', 0.005054440501650485, 'High', 0.0, 0.0, 1.0, 0]
Michael Ramone ['Human', 0.0047388505190610886, 'High', 0.0, 0.0, 1.0, 0]
Jennifer Mccormick ['Human', 0.004622395616024733, 'High', 0.0, 0.0, 1.0, 0]
Jen

/var/folders/17/gz6b52jj5cvfsq17xw_vghsh0000gn/T/ipykernel_25992/2497770618.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'unable to run' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  race_df.loc[idx, ['AI_label', 'assistance_score', 'confidence', 'fraction_ai', 'fraction_ai_assisted', 'fraction_human', 'num_ai_segments']] = result
/var/folders/17/gz6b52jj5cvfsq17xw_vghsh0000gn/T/ipykernel_25992/2497770618.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'unable to run' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  race_df.loc[idx, ['AI_label', 'assistance_score', 'confidence', 'fraction_ai', 'fraction_ai_assisted', 'fraction_human', 'num_ai_segments']] = result
/var/folders/17/gz6b52jj5cvfsq17xw_vghsh0000gn/T/ipykernel_25992/2

Andrew Bailey ['Human', 0.02854742482304573, 'High', 0.0, 0.0, 1.0, 0]
Elad Gross ['Human', 0.004487310536205769, 'High', 0.0, 0.0, 1.0, 0]
Elad Gross ['Human', 0.01693745329976082, 'High', 0.0, 0.0, 1.0, 0]
Austin Knudsen ['Human', 0.0043084025382995605, 'High', 0.0, 0.0, 1.0, 0]
Dan Bishop ['Human', 0.017700625583529472, 'High', 0.0, 0.0, 1.0, 0]
Dan Bishop ['Human', 0.0063428329303860664, 'High', 0.0, 0.0, 1.0, 0]
Dan Bishop ['Human', 0.010111949406564236, 'High', 0.0, 0.0, 1.0, 0]
Jeff Jackson ['Human', 0.006451108958572149, 'High', 0.0, 0.0, 1.0, 0]
Dan Rayfield ['Human', 0.005438329428090978, 'High', 0.0, 0.0, 1.0, 0]
Will Lathrop ['Human', 0.005409943405538797, 'High', 0.0, 0.0, 1.0, 0]
Dave Sunday ['Human', 0.004397788550704718, 'High', 0.0, 0.0, 1.0, 0]
Dave Sunday ['Human', 0.007755926204923704, 'High', 0.0, 0.0, 1.0, 0]
Eugene Depasquale ['Human', 0.005412740632891655, 'High', 0.0, 0.0, 1.0, 0]
Eugene Depasquale ['Human', 0.03664596416314857, 'Medium', 0.0, 0.0, 1.0, 0]
Dere

In [90]:
combined_80.to_csv("combined_80.csv")

In [92]:
combined_80['cleaned_text'].str.split().str.len().mean()

697.9872727272727

In [98]:
#run it on the entire text
combined_100 = pd.concat([gov_added, ag_added, mayor_added, us_senate_added], ignore_index=True)

In [100]:
check_entire_race(combined_100)

Andy Beshear ['Human', 0.005622072145342827, 'High', 0.0, 0.0, 1.0, 0]


/var/folders/17/gz6b52jj5cvfsq17xw_vghsh0000gn/T/ipykernel_25992/133460498.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Human' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  race_df.loc[idx, ['AI_label', 'assistance_score', 'confidence', 'fraction_ai', 'fraction_ai_assisted', 'fraction_human', 'num_ai_segments']] = result
/var/folders/17/gz6b52jj5cvfsq17xw_vghsh0000gn/T/ipykernel_25992/133460498.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'High' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  race_df.loc[idx, ['AI_label', 'assistance_score', 'confidence', 'fraction_ai', 'fraction_ai_assisted', 'fraction_human', 'num_ai_segments']] = result


Daniel Cameron ['Human', 0.01741171291048106, 'High', 0.0, 0.0, 1.0, 0]
Daniel Cameron ['AI', 0.8537339926549197, 'Medium', 1.0, 0.0, 0.0, 2]
Daniel Cameron ['AI', 0.9857575063194548, 'High', 1.0, 0.0, 0.0, 2]
Daniel Cameron ['Human', 0.28203206396467884, 'Low', 0.0, 0.0, 1.0, 0]
Jeff Landry ['Human', 0.006908764435838048, 'High', 0.0, 0.0, 1.0, 0]
Jeff Landry ['Human', 0.009802112141476263, 'High', 0.0, 0.0, 1.0, 0]
Brandon Presley ['Human', 0.004862438204110644, 'High', 0.0, 0.0, 1.0, 0]
Tate Reeves ['Human', 0.005038095638155937, 'High', 0.0, 0.0, 1.0, 0]
Tate Reeves ['Human', 0.005264014471322298, 'High', 0.0, 0.0, 1.0, 0]
Matt Meyer ['Human', 0.008963912827727783, 'High', 0.0, 0.0, 1.0, 0]
Michael Ramone ['Human', 0.007256772369146347, 'High', 0.0, 0.0, 1.0, 0]
Michael Ramone ['Human', 0.006030693394455334, 'High', 0.0, 0.0, 1.0, 0]
Michael Ramone ['Human', 0.00632522787200287, 'High', 0.0, 0.0, 1.0, 0]
Jennifer Mccormick ['Human', 0.005547715350985527, 'High', 0.0, 0.0, 1.0, 0]
J

/var/folders/17/gz6b52jj5cvfsq17xw_vghsh0000gn/T/ipykernel_25992/133460498.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'unable to run' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  race_df.loc[idx, ['AI_label', 'assistance_score', 'confidence', 'fraction_ai', 'fraction_ai_assisted', 'fraction_human', 'num_ai_segments']] = result
/var/folders/17/gz6b52jj5cvfsq17xw_vghsh0000gn/T/ipykernel_25992/133460498.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'unable to run' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  race_df.loc[idx, ['AI_label', 'assistance_score', 'confidence', 'fraction_ai', 'fraction_ai_assisted', 'fraction_human', 'num_ai_segments']] = result
/var/folders/17/gz6b52jj5cvfsq17xw_vghsh0000gn/T/ipykernel_25992/133

Bruce Harrell ['Human', 0.005818919142104713, 'High', 0.0, 0.0, 1.0, 0]
Katie Wilson ['Human', 0.02058959292480722, 'High', 0.0, 0.0, 1.0, 0]
Kari Lake ['Human', 0.018480936708372264, 'High', 0.0, 0.0, 1.0, 0]
Kari Lake ['Human', 0.0064066625200212, 'High', 0.0, 0.0, 1.0, 0]
Kari Lake ['Human', 0.01179938337632588, 'High', 0.0, 0.0, 1.0, 0]
Ruben Gallego ['Human', 0.005455920231755015, 'High', 0.0, 0.0, 1.0, 0]
Ruben Gallego ['Human', 0.012681391090154648, 'High', 0.0, 0.0, 1.0, 0]
Ruben Gallego ['Human', 0.004696518182754517, 'High', 0.0, 0.0, 1.0, 0]
Ruben Gallego ['Human', 0.015933264046907425, 'High', 0.0, 0.0, 1.0, 0]
Ruben Gallego ['Human', 0.004739276133477688, 'High', 0.0, 0.0, 1.0, 0]
Ruben Gallego ['Human', 0.005614593866214688, 'High', 0.0, 0.0, 1.0, 0]
Ruben Gallego ['Human', 0.007262190466281027, 'High', 0.0, 0.0, 1.0, 0]
Ruben Gallego ['Human', 0.005546203814446926, 'High', 0.0, 0.0, 1.0, 0]
Adam B. Schiff ['Human', 0.004961222720642885, 'High', 0.0, 0.0, 1.0, 0]
Adam B. 

In [102]:
combined_100.to_csv("combined_100.csv")